In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
!git clone https://github.com/spMohanty/PlantVillage-Dataset.git

Cloning into 'PlantVillage-Dataset'...
remote: Enumerating objects: 163264, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 163264 (delta 16), reused 25 (delta 9), pack-reused 163229 (from 1)
Receiving objects: 100% (163264/163264), 2.00 GiB | 39.62 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Updating files: 100% (182404/182404), done.


In [ ]:
DATA_PATH = "/content/PlantVillage-Dataset/raw/color"

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 20

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)
num_classes = len(train_ds.class_names)

print("✅ Num classes:", num_classes)

Found 54306 files belonging to 38 classes.
Using 43445 files for training.
Found 54306 files belonging to 38 classes.
Using 10861 files for validation.
✅ Num classes: 38


In [ ]:
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
])

norm = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (norm(data_aug(x)), y))
val_ds   = val_ds.map(lambda x, y: (norm(x), y))

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(512, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),  # 🔥 tốt hơn Flatten

    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(num_classes, activation='softmax')
])


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3)
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 174s 231ms/step - accuracy: 0.6605 - loss: 1.1740 - val_accuracy: 0.4916 - val_loss: 2.0417 - learning_rate: 3.0000e-04
Epoch 2/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 141s 208ms/step - accuracy: 0.8642 - loss: 0.4453 - val_accuracy: 0.6754 - val_loss: 1.1110 - learning_rate: 3.0000e-04
Epoch 3/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 141s 208ms/step - accuracy: 0.9160 - loss: 0.2704 - val_accuracy: 0.6609 - val_loss: 1.2151 - learning_rate: 3.0000e-04
Epoch 4/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 139s 204ms/step - accuracy: 0.9354 - loss: 0.2012 - val_accuracy: 0.8206 - val_loss: 0.5967 - learning_rate: 3.0000e-04
Epoch 5/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 145s 214ms/step - accuracy: 0.9490 - loss: 0.1625 - val_accuracy: 0.8706 - val_loss: 0.4145 - learning_rate: 3.0000e-04
Epoch 6/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 144s 212ms/step - accuracy: 0.9596 - loss: 0.1284 - val_accuracy: 0.8522 - val_loss: 0.4920 - learning_rate: 3.0000e-04
Epoch 7/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
loss, acc = model.evaluate(val_ds)

170/170 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.9678 - loss: 0.0955
